In [1]:
import sys
import os

sys.path.append(os.path.abspath("20260405-172134/f1_prediction"))
print(os.path.abspath("20260405-172134/f1_prediction"))
print(os.listdir("20260405-172134/f1_prediction")[:20])

/datasets/_deepnote_work/20260405-172134/f1_prediction
['.DS_Store', '__pycache__', 'data_loader.py', 'feature_engineering.py', 'main.py', 'output', 'quali_model.py', 'race_simulation.py', 'requirements.txt', 'standings.py']


In [2]:
from data_loader import (
    load_master_data,
    load_all_priors,
    load_2026_all_laptimes,
    load_2026_qualifying_results,
)

from feature_engineering import (
    build_training_features,
    build_2026_features,
    compute_2026_pace_signals,
    aggregate_2026_pace,
)

from quali_model import QualiModel, predict_all_grids
from race_simulation import simulate_race, simulate_season, expected_points_from_simulation
from standings import compute_expected_standings


[INFO] LightGBM/XGBoost unavailable. Using scikit-learn GradientBoosting.


In [3]:
master = load_master_data(min_year=2018)
priors = load_all_priors()
laps_2026 = load_2026_all_laptimes()
aus_q_results = load_2026_qualifying_results()

print("master shape:", master.shape)
print("driver priors:", priors["driver_priors"].shape)
print("team priors:", priors["team_priors"].shape)
print("2026 laps:", laps_2026.shape)
print("AUS quali:", aus_q_results.shape)

[INFO] Master dataset not found – building from converted_json CSVs.
[WARNING] converted_json not found either – no historical training data.
[WARNING] 2026 data directory not found: /datasets/_deepnote_work/20260405-172134/2026 cleaned
master shape: (0, 0)
driver priors: (0, 0)
team priors: (0, 0)
2026 laps: (0, 0)
AUS quali: (0, 0)


In [4]:
pace_per_session = compute_2026_pace_signals(laps_2026)
pace_2026 = aggregate_2026_pace(pace_per_session)

print("pace_per_session:", pace_per_session.shape)
print("pace_2026:", pace_2026.shape)
pace_2026.head()

pace_per_session: (0, 0)
pace_2026: (0, 3)


,driver,pace_2026_quali,pace_2026_race


In [5]:
import pandas as pd

BASE = "f1_cleaned_output/priors"

priors = {
    "driver_priors": pd.read_csv(f"{BASE}/driver_priors_2026.csv"),
    "team_priors": pd.read_csv(f"{BASE}/team_priors_2026.csv"),
    "engine_priors": pd.read_csv(f"{BASE}/engine_priors_2026.csv"),
    "track_priors": pd.read_csv(f"{BASE}/track_priors_2026.csv"),
    "reliability_priors": pd.read_csv(f"{BASE}/reliability_priors_2026.csv"),
    "upgrade_priors": pd.read_csv(f"{BASE}/upgrade_priors_2026.csv"),
    "team_engine_map": pd.read_csv(f"{BASE}/team_engine_map_2026.csv"),
    "driver_team_map": pd.read_csv(f"{BASE}/driver_team_map_2026.csv"),
    "weather": pd.read_csv("weather_all_tracks.csv"),
}

print(priors["driver_priors"].shape)
print(priors["team_priors"].shape)
print(priors["driver_priors"].head())
print(priors["team_priors"].head())

(22, 10)
(11, 11)
            driver             team  quali_skill  race_skill  wet_skill  \
0   Max Verstappen  Red Bull Racing         0.99        0.99       0.97   
1      Isak Hadjar  Red Bull Racing         0.86        0.88       0.82   
2     Lando Norris          McLaren         0.90        0.93       0.93   
3    Oscar Piastri          McLaren         0.93        0.95       0.90   
4  Charles Leclerc          Ferrari         1.00        0.97       0.92   

   tyre_management  start_skill  overtake_skill  defense_skill  consistency  
0             0.97         0.93            0.97           0.97         0.98  
1             0.83         0.85            0.84           0.83         0.85  
2             0.94         0.92            0.93           0.92         0.92  
3             0.92         0.92            0.91           0.91         0.90  
4             0.94         1.00            0.97           0.96         0.97  
              team engine_supplier  base_race_pace  base_quali_

In [6]:
features_2026 = build_2026_features(
    driver_priors=priors["driver_priors"],
    team_priors=priors["team_priors"],
    engine_priors=priors["engine_priors"],
    track_priors=priors["track_priors"],
    reliability_priors=priors["reliability_priors"],
    upgrade_priors=priors["upgrade_priors"],
    team_engine_map=priors["team_engine_map"],
    driver_team_map=priors["driver_team_map"],
    weather=priors["weather"],
    pace_2026=pace_2026,
)

print("features_2026:", features_2026.shape)
features_2026.head()

features_2026: (484, 47)


,round,circuit,driver,team,engine_supplier,driver_strength,wet_strength,race_skill,tyre_mgmt,start_skill,...,dnf_stress,street_factor,weather_variability,laps,avg_temp,rain_flag,temp_norm,pace_x_power,pace_x_downforce,wet_x_rain
0,1,Australia,Max Verstappen,Red Bull Racing,Red Bull Ford,0.99,0.97,0.99,0.97,0.93,...,0.45,0.4,0.5,58,10.904167,1.0,0.272604,0.504,0.588,0.97
1,1,Australia,Isak Hadjar,Red Bull Racing,Red Bull Ford,0.86,0.82,0.88,0.83,0.85,...,0.45,0.4,0.5,58,10.904167,1.0,0.272604,0.504,0.588,0.82
2,1,Australia,Lando Norris,McLaren,Mercedes,0.90,0.93,0.93,0.94,0.92,...,0.45,0.4,0.5,58,10.904167,1.0,0.272604,0.552,0.616,0.93
3,1,Australia,Oscar Piastri,McLaren,Mercedes,0.93,0.90,0.95,0.92,0.92,...,0.45,0.4,0.5,58,10.904167,1.0,0.272604,0.552,0.616,0.90
4,1,Australia,Charles Leclerc,Ferrari,Ferrari,1.00,0.92,0.97,0.94,1.00,...,0.45,0.4,0.5,58,10.904167,1.0,0.272604,0.564,0.679,0.92


In [7]:
train_df = build_training_features(
    master=master,
    track_priors=priors["track_priors"],
    pace_2026=pace_2026,
)

print("train_df:", train_df.shape)
train_df.head()

train_df: (0, 0)


""


In [8]:
qm = QualiModel()
qm.train(train_df, verbose=True)

[WARNING] No training data – will use prior + 2026 scoring.


In [9]:
known_grids = {}
if not aus_q_results.empty:
    known_grids[1] = aus_q_results

all_quali = predict_all_grids(
    qm,
    features_2026,
    seed=2026,
    known_grids=known_grids,
)

print("all_quali:", all_quali.shape)
all_quali.head(20)

all_quali: (484, 10)


,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,George Russell,Mercedes,Mercedes,1.046562,1,Australia,1,1.0,10.904167,0.5
1,Kimi Antonelli,Mercedes,Mercedes,1.019292,2,Australia,1,1.0,10.904167,0.5
2,Lewis Hamilton,Ferrari,Ferrari,1.018664,3,Australia,1,1.0,10.904167,0.5
3,Charles Leclerc,Ferrari,Ferrari,1.009957,4,Australia,1,1.0,10.904167,0.5
4,Oscar Piastri,McLaren,Mercedes,0.999006,5,Australia,1,1.0,10.904167,0.5
5,Max Verstappen,Red Bull Racing,Red Bull Ford,0.979171,6,Australia,1,1.0,10.904167,0.5
6,Lando Norris,McLaren,Mercedes,0.965136,7,Australia,1,1.0,10.904167,0.5
7,Carlos Sainz,Williams,Mercedes,0.961960,8,Australia,1,1.0,10.904167,0.5
8,Oliver Bearman,Haas,Ferrari,0.951552,9,Australia,1,1.0,10.904167,0.5
9,Franco Colapinto,Alpine,Mercedes,0.949496,10,Australia,1,1.0,10.904167,0.5


In [10]:
round_1 = simulate_race(
    features_2026=features_2026,
    round_num=1,
    n_simulations=3000,
    seed=2026,
    qualifying_grid=all_quali,
)

round_1["summary"].head(22)

,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,George Russell,Mercedes,0.2557,0.6193,0.9417,0.0570,3.96
1,2,Kimi Antonelli,Mercedes,0.2343,0.6180,0.9377,0.0607,4.08
2,3,Charles Leclerc,Ferrari,0.1940,0.5443,0.9653,0.0327,3.99
3,4,Lewis Hamilton,Ferrari,0.1833,0.5640,0.9627,0.0343,3.93
4,5,Oscar Piastri,McLaren,0.0500,0.2523,0.8990,0.0877,6.29
5,6,Lando Norris,McLaren,0.0460,0.2133,0.8957,0.0853,6.52
6,7,Max Verstappen,Red Bull Racing,0.0323,0.1480,0.8143,0.1620,8.07
7,8,Carlos Sainz,Williams,0.0020,0.0223,0.7203,0.1063,9.66
8,9,Oliver Bearman,Haas,0.0017,0.0070,0.5457,0.1270,11.27
9,10,Pierre Gasly,Alpine,0.0003,0.0053,0.5240,0.1100,11.27


In [11]:
driver_pts = expected_points_from_simulation(round_1, n_simulations=3000)
driver_pts.head(15)

,driver,expected_points
0,George Russell,15.853
1,Kimi Antonelli,15.585
2,Lewis Hamilton,14.920
3,Charles Leclerc,14.786
4,Oscar Piastri,9.967
5,Lando Norris,9.448
6,Max Verstappen,7.756
7,Carlos Sainz,3.893
8,Oliver Bearman,2.201
9,Pierre Gasly,2.105


In [12]:
season_results = {}

for rnd in sorted(features_2026["round"].unique()):
    print(f"Simulating round {rnd}...")

    race_result = simulate_race(
        features_2026=features_2026,
        round_num=rnd,
        n_simulations=1000,
        seed=2026 + rnd,
        qualifying_grid=all_quali,
    )

    season_results[rnd] = race_result

Simulating round 1...
Simulating round 2...
Simulating round 3...
Simulating round 4...
Simulating round 5...
Simulating round 6...
Simulating round 7...
Simulating round 8...
Simulating round 9...
Simulating round 10...
Simulating round 11...
Simulating round 12...
Simulating round 13...
Simulating round 14...
Simulating round 15...
Simulating round 16...
Simulating round 17...
Simulating round 18...
Simulating round 19...
Simulating round 20...
Simulating round 21...
Simulating round 22...


In [13]:
ROUND_TO_VIEW = 1
DRIVER_TO_VIEW = "Max Verstappen"
TEAM_TO_VIEW = "Mercedes"

In [15]:
print(f"Driver View: {DRIVER_TO_VIEW}")

driver_probs = []
for rnd in sorted(k for k in season_results.keys() if isinstance(k, int)):
    probs = season_results[rnd].get("probabilities", pd.DataFrame())
    if probs.empty:
        continue
    row = probs[probs["driver"] == DRIVER_TO_VIEW].copy()
    if not row.empty:
        row["round"] = rnd
        driver_probs.append(row)

if driver_probs:
    driver_probs_df = pd.concat(driver_probs, ignore_index=True)
    cols = [c for c in [
        "round",
        "driver",
        "team",
        "win_prob",
        "podium_prob",
        "top5_prob",
        "points_prob",
        "dnf_prob",
        "expected_pos"
    ] if c in driver_probs_df.columns]
    display(driver_probs_df[cols].sort_values("round").reset_index(drop=True))
else:
    print("No driver probability rows found.")

Driver View: Max Verstappen
No driver probability rows found.


In [16]:
ROUND_TO_VIEW = 3

In [17]:
print(f"ROUND {ROUND_TO_VIEW} QUALIFYING")
display(
    all_quali[all_quali["round"] == ROUND_TO_VIEW]
    .sort_values("quali_pos")
    .reset_index(drop=True)
)

ROUND 3 QUALIFYING


,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,Charles Leclerc,Ferrari,Ferrari,1.073588,1,Japan,3,1.0,24.425,0.55
1,Kimi Antonelli,Mercedes,Mercedes,1.055791,2,Japan,3,1.0,24.425,0.55
2,George Russell,Mercedes,Mercedes,1.021716,3,Japan,3,1.0,24.425,0.55
3,Oscar Piastri,McLaren,Mercedes,1.018514,4,Japan,3,1.0,24.425,0.55
4,Lando Norris,McLaren,Mercedes,0.996248,5,Japan,3,1.0,24.425,0.55
5,Carlos Sainz,Williams,Mercedes,0.970267,6,Japan,3,1.0,24.425,0.55
6,Gabriel Bortoleto,Audi,Audi,0.958533,7,Japan,3,1.0,24.425,0.55
7,Lewis Hamilton,Ferrari,Ferrari,0.955633,8,Japan,3,1.0,24.425,0.55
8,Max Verstappen,Red Bull Racing,Red Bull Ford,0.954484,9,Japan,3,1.0,24.425,0.55
9,Franco Colapinto,Alpine,Mercedes,0.948696,10,Japan,3,1.0,24.425,0.55


In [18]:
print(f"ROUND {ROUND_TO_VIEW} RACE SUMMARY")
display(season_results[ROUND_TO_VIEW]["summary"])

ROUND 3 RACE SUMMARY


,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,Kimi Antonelli,Mercedes,0.258,0.642,0.936,0.063,3.98
1,2,George Russell,Mercedes,0.255,0.613,0.941,0.057,3.99
2,3,Charles Leclerc,Ferrari,0.244,0.636,0.957,0.042,3.68
3,4,Lewis Hamilton,Ferrari,0.124,0.483,0.970,0.029,4.23
4,5,Lando Norris,McLaren,0.054,0.235,0.904,0.093,6.34
5,6,Oscar Piastri,McLaren,0.047,0.235,0.909,0.080,6.16
6,7,Max Verstappen,Red Bull Racing,0.016,0.120,0.823,0.150,8.18
7,8,Oliver Bearman,Haas,0.001,0.007,0.503,0.152,11.65
8,9,Alex Albon,Williams,0.001,0.004,0.410,0.109,12.09
9,10,Sergio Perez,Cadillac,0.000,0.000,0.096,0.178,15.36


In [19]:
print(f"Team View: {TEAM_TO_VIEW}")

team_probs = []

for rnd in sorted(k for k in season_results.keys() if isinstance(k, int)):
    probs = season_results[rnd].get("probabilities", pd.DataFrame())
    if probs.empty:
        continue

    rows = probs[probs["team"] == TEAM_TO_VIEW].copy()
    if not rows.empty:
        rows["round"] = rnd
        team_probs.append(rows)

if team_probs:
    team_probs_df = pd.concat(team_probs, ignore_index=True)

    cols = [c for c in [
        "round",
        "driver",
        "team",
        "win_prob",
        "podium_prob",
        "top5_prob",
        "points_prob",
        "dnf_prob",
        "expected_pos"
    ] if c in team_probs_df.columns]

    display(
        team_probs_df[cols]
        .sort_values(["round", "expected_pos"])
        .reset_index(drop=True)
    )
else:
    print("No team probability rows found.")

Team View: Mercedes
No team probability rows found.


In [20]:
for rnd in sorted(k for k in season_results.keys() if isinstance(k, int)):
    probs = season_results[rnd].get("probabilities", pd.DataFrame())
    if not probs.empty:
        print(sorted(probs["team"].dropna().unique()))
        break

In [22]:
ROUND_TO_VIEW = 1

In [23]:
for rnd in sorted(season_results.keys()):
    print(f"\n========== ROUND {rnd} RACE SUMMARY ==========")
    summary = season_results[rnd].get("summary", pd.DataFrame())
    if not summary.empty:
        display(summary.reset_index(drop=True))
    else:
        print("No summary found.")


========== ROUND 1 RACE SUMMARY ==========


,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,Kimi Antonelli,Mercedes,0.254,0.629,0.952,0.047,3.78
1,2,George Russell,Mercedes,0.243,0.624,0.950,0.048,3.84
2,3,Lewis Hamilton,Ferrari,0.203,0.539,0.962,0.037,4.00
3,4,Charles Leclerc,Ferrari,0.186,0.569,0.970,0.027,3.82
4,5,Oscar Piastri,McLaren,0.042,0.235,0.911,0.078,6.30
5,6,Max Verstappen,Red Bull Racing,0.035,0.162,0.829,0.142,7.70
6,7,Lando Norris,McLaren,0.035,0.204,0.896,0.082,6.60
7,8,Isak Hadjar,Red Bull Racing,0.001,0.002,0.276,0.172,13.40
8,9,Carlos Sainz,Williams,0.001,0.018,0.691,0.128,10.03
9,10,Oliver Bearman,Haas,0.000,0.005,0.528,0.104,11.19



========== ROUND 2 RACE SUMMARY ==========


,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,George Russell,Mercedes,0.269,0.626,0.942,0.057,3.96
1,2,Kimi Antonelli,Mercedes,0.268,0.662,0.942,0.056,3.80
2,3,Charles Leclerc,Ferrari,0.222,0.648,0.973,0.025,3.47
3,4,Lewis Hamilton,Ferrari,0.135,0.476,0.968,0.028,4.28
4,5,Oscar Piastri,McLaren,0.061,0.280,0.906,0.087,6.07
5,6,Lando Norris,McLaren,0.027,0.183,0.875,0.102,6.99
6,7,Max Verstappen,Red Bull Racing,0.015,0.084,0.773,0.178,8.94
7,8,Carlos Sainz,Williams,0.003,0.017,0.651,0.104,10.04
8,9,Oliver Bearman,Haas,0.000,0.009,0.526,0.137,11.34
9,10,Sergio Perez,Cadillac,0.000,0.000,0.099,0.177,15.29



========== ROUND 3 RACE SUMMARY ==========


...

,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,Kimi Antonelli,Mercedes,0.294,0.668,0.937,0.060,3.85
1,2,George Russell,Mercedes,0.259,0.607,0.928,0.072,4.24
2,3,Charles Leclerc,Ferrari,0.180,0.568,0.965,0.030,3.94
3,4,Lewis Hamilton,Ferrari,0.118,0.446,0.958,0.032,4.52
4,5,Oscar Piastri,McLaren,0.056,0.263,0.892,0.088,6.26
5,6,Max Verstappen,Red Bull Racing,0.044,0.196,0.828,0.146,7.51
6,7,Lando Norris,McLaren,0.041,0.209,0.891,0.086,6.71
7,8,Isak Hadjar,Red Bull Racing,0.004,0.013,0.550,0.165,11.32
8,9,Carlos Sainz,Williams,0.002,0.017,0.659,0.093,10.10
9,10,Liam Lawson,Racing Bulls,0.001,0.001,0.117,0.149,14.81



========== ROUND 21 RACE SUMMARY ==========


,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,George Russell,Mercedes,0.288,0.716,0.930,0.070,3.82
1,2,Kimi Antonelli,Mercedes,0.265,0.664,0.926,0.074,3.95
2,3,Charles Leclerc,Ferrari,0.219,0.624,0.961,0.039,3.66
3,4,Lewis Hamilton,Ferrari,0.169,0.535,0.965,0.034,4.02
4,5,Oscar Piastri,McLaren,0.033,0.192,0.897,0.098,6.41
5,6,Max Verstappen,Red Bull Racing,0.016,0.125,0.825,0.167,8.02
6,7,Lando Norris,McLaren,0.010,0.135,0.904,0.091,6.79
7,8,Esteban Ocon,Haas,0.000,0.000,0.277,0.148,13.18
8,9,Sergio Perez,Cadillac,0.000,0.000,0.034,0.180,16.25
9,10,Franco Colapinto,Alpine,0.000,0.000,0.220,0.104,13.38



========== ROUND 22 RACE SUMMARY ==========


,predicted_pos,driver,team,win_prob,podium_prob,points_prob,dnf_prob,expected_pos
0,1,George Russell,Mercedes,0.289,0.677,0.930,0.070,3.89
1,2,Kimi Antonelli,Mercedes,0.277,0.689,0.950,0.050,3.61
2,3,Lewis Hamilton,Ferrari,0.167,0.508,0.967,0.033,4.04
3,4,Charles Leclerc,Ferrari,0.166,0.568,0.959,0.041,3.95
4,5,Oscar Piastri,McLaren,0.053,0.232,0.925,0.069,5.89
5,6,Lando Norris,McLaren,0.043,0.224,0.918,0.073,6.06
6,7,Max Verstappen,Red Bull Racing,0.005,0.092,0.816,0.152,8.32
7,8,Esteban Ocon,Haas,0.000,0.000,0.327,0.138,12.77
8,9,Sergio Perez,Cadillac,0.000,0.000,0.040,0.163,16.00
9,10,Franco Colapinto,Alpine,0.000,0.001,0.247,0.125,13.40


In [24]:
for rnd in sorted(season_results.keys()):
    print(f"\n========== ROUND {rnd} FULL PROBABILITIES ==========")
    probs = season_results[rnd].get("probabilities", pd.DataFrame())
    if not probs.empty:
        display(probs.sort_values("predicted_pos").reset_index(drop=True))
    else:
        print("No probabilities found.")


========== ROUND 1 FULL PROBABILITIES ==========


,driver,team,win_prob,podium_prob,top5_prob,points_prob,dnf_prob,fl_prob,expected_pos,avg_pace,predicted_pos
0,Kimi Antonelli,Mercedes,0.254,0.629,0.853,0.952,0.047,0.254,3.78,1.011227,1
1,George Russell,Mercedes,0.243,0.624,0.851,0.950,0.048,0.243,3.84,1.010698,2
2,Lewis Hamilton,Ferrari,0.203,0.539,0.827,0.962,0.037,0.203,4.00,1.005251,3
3,Charles Leclerc,Ferrari,0.186,0.569,0.827,0.970,0.027,0.186,3.82,1.005609,4
4,Oscar Piastri,McLaren,0.042,0.235,0.531,0.911,0.078,0.042,6.30,0.980517,5
5,Max Verstappen,Red Bull Racing,0.035,0.162,0.420,0.829,0.142,0.035,7.70,0.973544,6
6,Lando Norris,McLaren,0.035,0.204,0.474,0.896,0.082,0.035,6.60,0.977640,7
7,Isak Hadjar,Red Bull Racing,0.001,0.002,0.014,0.276,0.172,0.001,13.40,0.902864,8
8,Carlos Sainz,Williams,0.001,0.018,0.088,0.691,0.128,0.001,10.03,0.938670,9
9,Oliver Bearman,Haas,0.000,0.005,0.031,0.528,0.104,0.000,11.19,0.921388,10



========== ROUND 2 FULL PROBABILITIES ==========


,driver,team,win_prob,podium_prob,top5_prob,points_prob,dnf_prob,fl_prob,expected_pos,avg_pace,predicted_pos
0,George Russell,Mercedes,0.269,0.626,0.846,0.942,0.057,0.269,3.96,0.997102,1
1,Kimi Antonelli,Mercedes,0.268,0.662,0.858,0.942,0.056,0.268,3.80,0.998412,2
2,Charles Leclerc,Ferrari,0.222,0.648,0.872,0.973,0.025,0.222,3.47,0.994652,3
3,Lewis Hamilton,Ferrari,0.135,0.476,0.770,0.968,0.028,0.135,4.28,0.984707,4
4,Oscar Piastri,McLaren,0.061,0.280,0.611,0.906,0.087,0.061,6.07,0.973887,5
5,Lando Norris,McLaren,0.027,0.183,0.472,0.875,0.102,0.027,6.99,0.964902,6
6,Max Verstappen,Red Bull Racing,0.015,0.084,0.287,0.773,0.178,0.015,8.94,0.954498,7
7,Carlos Sainz,Williams,0.003,0.017,0.092,0.651,0.104,0.003,10.04,0.929263,8
8,Oliver Bearman,Haas,0.000,0.009,0.053,0.526,0.137,0.000,11.34,0.919576,9
9,Sergio Perez,Cadillac,0.000,0.000,0.002,0.099,0.177,0.000,15.29,0.879879,10



========== ROUND 3 FULL PROBABILITIES ==========


...

,driver,team,win_prob,podium_prob,top5_prob,points_prob,dnf_prob,fl_prob,expected_pos,avg_pace,predicted_pos
0,Kimi Antonelli,Mercedes,0.294,0.668,0.840,0.937,0.060,0.294,3.85,1.022288,1
1,George Russell,Mercedes,0.259,0.607,0.829,0.928,0.072,0.259,4.24,1.019760,2
2,Charles Leclerc,Ferrari,0.180,0.568,0.819,0.965,0.030,0.180,3.94,1.013521,3
3,Lewis Hamilton,Ferrari,0.118,0.446,0.742,0.958,0.032,0.118,4.52,1.005544,4
4,Oscar Piastri,McLaren,0.056,0.263,0.562,0.892,0.088,0.056,6.26,0.994345,5
5,Max Verstappen,Red Bull Racing,0.044,0.196,0.464,0.828,0.146,0.044,7.51,0.991231,6
6,Lando Norris,McLaren,0.041,0.209,0.482,0.891,0.086,0.041,6.71,0.988553,7
7,Isak Hadjar,Red Bull Racing,0.004,0.013,0.057,0.550,0.165,0.004,11.32,0.945898,8
8,Carlos Sainz,Williams,0.002,0.017,0.078,0.659,0.093,0.002,10.10,0.950968,9
9,Liam Lawson,Racing Bulls,0.001,0.001,0.003,0.117,0.149,0.001,14.81,0.906557,10



========== ROUND 21 FULL PROBABILITIES ==========


,driver,team,win_prob,podium_prob,top5_prob,points_prob,dnf_prob,fl_prob,expected_pos,avg_pace,predicted_pos
0,George Russell,Mercedes,0.288,0.716,0.879,0.930,0.070,0.288,3.82,1.050917,1
1,Kimi Antonelli,Mercedes,0.265,0.664,0.879,0.926,0.074,0.265,3.95,1.049544,2
2,Charles Leclerc,Ferrari,0.219,0.624,0.881,0.961,0.039,0.219,3.66,1.044354,3
3,Lewis Hamilton,Ferrari,0.169,0.535,0.837,0.965,0.034,0.169,4.02,1.037697,4
4,Oscar Piastri,McLaren,0.033,0.192,0.556,0.897,0.098,0.033,6.41,1.016415,5
5,Max Verstappen,Red Bull Racing,0.016,0.125,0.396,0.825,0.167,0.016,8.02,1.007798,6
6,Lando Norris,McLaren,0.010,0.135,0.452,0.904,0.091,0.010,6.79,1.009272,7
7,Esteban Ocon,Haas,0.000,0.000,0.004,0.277,0.148,0.000,13.18,0.936041,8
8,Sergio Perez,Cadillac,0.000,0.000,0.001,0.034,0.180,0.000,16.25,0.900865,9
9,Franco Colapinto,Alpine,0.000,0.000,0.002,0.220,0.104,0.000,13.38,0.928990,10



========== ROUND 22 FULL PROBABILITIES ==========


,driver,team,win_prob,podium_prob,top5_prob,points_prob,dnf_prob,fl_prob,expected_pos,avg_pace,predicted_pos
0,George Russell,Mercedes,0.289,0.677,0.870,0.930,0.070,0.289,3.89,1.041124,1
1,Kimi Antonelli,Mercedes,0.277,0.689,0.872,0.950,0.050,0.277,3.61,1.039697,2
2,Lewis Hamilton,Ferrari,0.167,0.508,0.831,0.967,0.033,0.167,4.04,1.027905,3
3,Charles Leclerc,Ferrari,0.166,0.568,0.861,0.959,0.041,0.166,3.95,1.031328,4
4,Oscar Piastri,McLaren,0.053,0.232,0.581,0.925,0.069,0.053,5.89,1.010387,5
5,Lando Norris,McLaren,0.043,0.224,0.561,0.918,0.073,0.043,6.06,1.008034,6
6,Max Verstappen,Red Bull Racing,0.005,0.092,0.298,0.816,0.152,0.005,8.32,0.992782,7
7,Esteban Ocon,Haas,0.000,0.000,0.009,0.327,0.138,0.000,12.77,0.935630,8
8,Sergio Perez,Cadillac,0.000,0.000,0.000,0.040,0.163,0.000,16.00,0.900116,9
9,Franco Colapinto,Alpine,0.000,0.001,0.006,0.247,0.125,0.000,13.40,0.927801,10


In [25]:
for rnd in sorted(all_quali["round"].unique()):
    print(f"\n========== ROUND {rnd} QUALIFYING ==========")
    display(
        all_quali[all_quali["round"] == rnd]
        .sort_values("quali_pos")
        .reset_index(drop=True)
    )


========== ROUND 1 QUALIFYING ==========


,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,George Russell,Mercedes,Mercedes,1.046562,1,Australia,1,1.0,10.904167,0.5
1,Kimi Antonelli,Mercedes,Mercedes,1.019292,2,Australia,1,1.0,10.904167,0.5
2,Lewis Hamilton,Ferrari,Ferrari,1.018664,3,Australia,1,1.0,10.904167,0.5
3,Charles Leclerc,Ferrari,Ferrari,1.009957,4,Australia,1,1.0,10.904167,0.5
4,Oscar Piastri,McLaren,Mercedes,0.999006,5,Australia,1,1.0,10.904167,0.5
5,Max Verstappen,Red Bull Racing,Red Bull Ford,0.979171,6,Australia,1,1.0,10.904167,0.5
6,Lando Norris,McLaren,Mercedes,0.965136,7,Australia,1,1.0,10.904167,0.5
7,Carlos Sainz,Williams,Mercedes,0.961960,8,Australia,1,1.0,10.904167,0.5
8,Oliver Bearman,Haas,Ferrari,0.951552,9,Australia,1,1.0,10.904167,0.5
9,Franco Colapinto,Alpine,Mercedes,0.949496,10,Australia,1,1.0,10.904167,0.5



========== ROUND 2 QUALIFYING ==========


,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,Charles Leclerc,Ferrari,Ferrari,1.029024,1,China,2,0.0,28.0375,0.45
1,Kimi Antonelli,Mercedes,Mercedes,1.011504,2,China,2,0.0,28.0375,0.45
2,George Russell,Mercedes,Mercedes,0.980958,3,China,2,0.0,28.0375,0.45
3,Oscar Piastri,McLaren,Mercedes,0.976709,4,China,2,0.0,28.0375,0.45
4,Esteban Ocon,Haas,Ferrari,0.942888,5,China,2,0.0,28.0375,0.45
5,Oliver Bearman,Haas,Ferrari,0.933808,6,China,2,0.0,28.0375,0.45
6,Arvid Lindblad,Racing Bulls,Red Bull Ford,0.931482,7,China,2,0.0,28.0375,0.45
7,Lewis Hamilton,Ferrari,Ferrari,0.928202,8,China,2,0.0,28.0375,0.45
8,Franco Colapinto,Alpine,Mercedes,0.911905,9,China,2,0.0,28.0375,0.45
9,Isak Hadjar,Red Bull Racing,Red Bull Ford,0.908409,10,China,2,0.0,28.0375,0.45



========== ROUND 3 QUALIFYING ==========


...

,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,Charles Leclerc,Ferrari,Ferrari,0.997562,1,Las Vegas,20,0.0,34.5875,0.25
1,George Russell,Mercedes,Mercedes,0.979403,2,Las Vegas,20,0.0,34.5875,0.25
2,Kimi Antonelli,Mercedes,Mercedes,0.954471,3,Las Vegas,20,0.0,34.5875,0.25
3,Oscar Piastri,McLaren,Mercedes,0.949107,4,Las Vegas,20,0.0,34.5875,0.25
4,Max Verstappen,Red Bull Racing,Red Bull Ford,0.940639,5,Las Vegas,20,0.0,34.5875,0.25
5,Lando Norris,McLaren,Mercedes,0.919243,6,Las Vegas,20,0.0,34.5875,0.25
6,Isak Hadjar,Red Bull Racing,Red Bull Ford,0.918446,7,Las Vegas,20,0.0,34.5875,0.25
7,Lewis Hamilton,Ferrari,Ferrari,0.917182,8,Las Vegas,20,0.0,34.5875,0.25
8,Oliver Bearman,Haas,Ferrari,0.900016,9,Las Vegas,20,0.0,34.5875,0.25
9,Carlos Sainz,Williams,Mercedes,0.893139,10,Las Vegas,20,0.0,34.5875,0.25



========== ROUND 21 QUALIFYING ==========


,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,George Russell,Mercedes,Mercedes,0.998221,1,Qatar,21,0.0,36.970833,0.2
1,Charles Leclerc,Ferrari,Ferrari,0.960534,2,Qatar,21,0.0,36.970833,0.2
2,Kimi Antonelli,Mercedes,Mercedes,0.955960,3,Qatar,21,0.0,36.970833,0.2
3,Lewis Hamilton,Ferrari,Ferrari,0.955456,4,Qatar,21,0.0,36.970833,0.2
4,Oscar Piastri,McLaren,Mercedes,0.953598,5,Qatar,21,0.0,36.970833,0.2
5,Max Verstappen,Red Bull Racing,Red Bull Ford,0.935876,6,Qatar,21,0.0,36.970833,0.2
6,Pierre Gasly,Alpine,Mercedes,0.935544,7,Qatar,21,0.0,36.970833,0.2
7,Lando Norris,McLaren,Mercedes,0.931686,8,Qatar,21,0.0,36.970833,0.2
8,Alex Albon,Williams,Mercedes,0.908574,9,Qatar,21,0.0,36.970833,0.2
9,Liam Lawson,Racing Bulls,Red Bull Ford,0.906671,10,Qatar,21,0.0,36.970833,0.2



========== ROUND 22 QUALIFYING ==========


,driver,team,engine_supplier,quali_score,quali_pos,circuit,round,rain_flag,avg_temp,weather_variability
0,George Russell,Mercedes,Mercedes,0.994313,1,Abu Dhabi,22,0.0,34.1375,0.2
1,Kimi Antonelli,Mercedes,Mercedes,0.974187,2,Abu Dhabi,22,0.0,34.1375,0.2
2,Oscar Piastri,McLaren,Mercedes,0.971813,3,Abu Dhabi,22,0.0,34.1375,0.2
3,Lando Norris,McLaren,Mercedes,0.947363,4,Abu Dhabi,22,0.0,34.1375,0.2
4,Lewis Hamilton,Ferrari,Ferrari,0.940773,5,Abu Dhabi,22,0.0,34.1375,0.2
5,Charles Leclerc,Ferrari,Ferrari,0.931470,6,Abu Dhabi,22,0.0,34.1375,0.2
6,Arvid Lindblad,Racing Bulls,Red Bull Ford,0.913778,7,Abu Dhabi,22,0.0,34.1375,0.2
7,Max Verstappen,Red Bull Racing,Red Bull Ford,0.899350,8,Abu Dhabi,22,0.0,34.1375,0.2
8,Oliver Bearman,Haas,Ferrari,0.897495,9,Abu Dhabi,22,0.0,34.1375,0.2
9,Carlos Sainz,Williams,Mercedes,0.891776,10,Abu Dhabi,22,0.0,34.1375,0.2


In [26]:
driver_df, con_df = compute_expected_standings(
    season_results,
    n_simulations=1000
)

In [27]:
import os

os.makedirs("deepnote_output", exist_ok=True)

driver_df.to_csv("deepnote_output/driver_standings.csv", index=False)
con_df.to_csv("deepnote_output/constructor_standings.csv", index=False)
all_quali.to_csv("deepnote_output/all_qualifying_predictions.csv", index=False)

for rnd in sorted(k for k in season_results.keys() if isinstance(k, int)):
    season_results[rnd]["summary"].to_csv(
        f"deepnote_output/race_summary_round_{rnd}.csv",
        index=False
    )
    season_results[rnd]["probabilities"].to_csv(
        f"deepnote_output/race_probabilities_round_{rnd}.csv",
        index=False
    )

print("Saved to deepnote_output/")

Saved to deepnote_output/


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=a48761f7-46b0-4a39-ac60-090ae77a0ccc' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>